In [2]:
!pip install folium


   ---------------------------------------- 3/3 [folium]



In [3]:
import folium 
import pandas as pd 
from folium.plugins import MarkerCluster 
from folium.plugins import MousePosition 
from folium.features import DivIcon 


In [4]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df=pd.read_csv(URL)
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [5]:
spacex_df=spacex_df[['Launch Site','Lat','Long','class']]
launch_sites_df=spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df=launch_sites_df[['Launch Site','Lat','Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [6]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map=folium.Map(location=nasa_coordinate,zoom_start=10)

In [7]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle=folium.Circle(nasa_coordinate,radius=1000,color='#d35400',fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker=folium.map.Marker(nasa_coordinate ,
                         icon=DivIcon(icon_size=(20,20),
                                      icon_anchor=(0,0),
                                      html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',))
site_map.add_child(circle)
site_map.add_child(marker)

In [8]:
site_map=folium.Map(location=nasa_coordinate,zoom_start=5)

for index, row in launch_sites_df.iterrows():
    coordinate=[row['Lat'],row['Long']]
    launch_site_name=row['Launch Site']

    circle=folium.Circle(coordinate, redius=1000,color='#d35400',fill=True).add_child(folium.Popup(launch_site_name))
    marker=folium.map.Marker(coordinate, 
                             icon=DivIcon(icon_size=(20,20),
                                          icon_anchor=(0,0),
                                          html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % launch_site_name))
    site_map.add_child(circle)
    site_map.add_child(marker)
site_map


In [9]:
marker_cluster=MarkerCluster()

In [10]:
def assign_marker_color(launch_class):
    if launch_class==1:
        return 'green' 
    else:
        return 'red' 

spacex_df['marker_color']=spacex_df['class'].apply(assign_marker_color)

spacex_df.tail(10)


,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [13]:
site_map.add_child(marker_cluster)

for index,record in spacex_df.iterrows():
    marker=folium.Marker(location=[record['Lat'],record['Long']],
                         icon=folium.Icon(color='white', icon_color=record['marker_color']))
    marker_cluster.add_child(marker)

site_map 

In [14]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [15]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [ ]:
launch_site_lat = 28.563197
launch_site_lon = -80.576820

coastline_lat = 28.56367
coastline_lon = -80.56763

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

print(f"Distance to coastline: {distance_coastline:.2f} km")

Distance to coastline: 0.90 km


In [17]:
coast_coord = [coastline_lat, coastline_lon]

# 2. إنشاء الـ Marker الخاص بالمسافة
distance_marker = folium.Marker(
    location=coast_coord,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)

# 3. إضافة الـ Marker للخريطة
site_map.add_child(distance_marker)

# 4. (خطوة إضافية مهمة) رسم الخط الواصل بين الموقعين عشان تظهر النتيجة بوضوح
lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], coast_coord], weight=1)
site_map.add_child(lines)

# عرض الخريطة
site_map

In [18]:
coordinates = [
    [launch_site_lat, launch_site_lon], 
    [coastline_lat, coastline_lon]
]

lines = folium.PolyLine(locations=coordinates, weight=1)

site_map.add_child(lines)

TODO: Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use MousePosition to find the their coordinates on the map first

In [20]:
highway_lat, highway_lon = 28.56321, -80.57079
launch_site_coord = [28.563197, -80.576820]
distance_highway=calculate_distance(launch_site_coord[0],launch_site_coord[1],highway_lat,highway_lon)
distance_marker=folium.Marker([highway_lat,highway_lon],
                              icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0),
                                          html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_highway))).add_to(site_map)
folium.PolyLine(locations=[launch_site_coord,[highway_lat,highway_lon]],weight=1).add_to(site_map)
site_map


In [22]:
# 1. الإحداثيات (تقريبية للسكة الحديد)
railway_lat, railway_lon = 28.57205, -80.58527

# 2. حساب المسافة
distance_railway = calculate_distance(launch_site_coord[0], launch_site_coord[1], railway_lat, railway_lon)

# 3. إضافة الـ Marker والـ Line
folium.Marker(
    [railway_lat, railway_lon],
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
    html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_railway))
).add_to(site_map)

folium.PolyLine(locations=[launch_site_coord, [railway_lat, railway_lon]], weight=1).add_to(site_map)
site_map

In [24]:
# 1. إحداثيات مدينة Titusville القريبة
city_lat, city_lon = 28.6115, -80.8077

# 2. حساب المسافة
distance_city = calculate_distance(launch_site_coord[0], launch_site_coord[1], city_lat, city_lon)

# 3. إضافة الـ Marker والـ Line
folium.Marker(
    [city_lat, city_lon],
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
    html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_city))
).add_to(site_map)

folium.PolyLine(locations=[launch_site_coord, [city_lat, city_lon]], weight=1).add_to(site_map)
site_map